# Lab 14 · Sắp xếp đoạn báo cáo, diễn đạt đúng mức và thẩm định một kết luận

**Lập trình xử lý dữ liệu · 2627-1 · Giờ thực hành · Bài 14**

Chọn **Save a copy in Drive** trước khi sửa.

Lab đi theo bài giảng Bài 14. Bạn sẽ sắp xếp một đoạn báo cáo để kết luận đứng trước, chọn lời diễn đạt đúng mức
cho các số liệu thật của Santiago, tách một con số gộp theo loại phòng để thấy ảnh hưởng của cơ cấu nhóm,
rồi thẩm định một kết luận về chủ nhà chuyên nghiệp bằng bốn bước của bài giảng.

**Tài liệu:** [Wilke, Fundamentals of Data Visualization, chương 29](https://clauswilke.com/dataviz/telling-a-story.html);
[Calling Bullshit](https://callingbullshit.org/syllabus.html) (Bergstrom và West).
Lab dùng thêm `groupby` và `merge` (Bài 5), `resample` và cách cắt theo mốc chụp (Bài 8),
`crosstab` và giá trị thiếu (Bài 10).

## Cách làm việc trong bài lab

- Điền các ô `TODO` và phần giải thích.
  `assert` kiểm tra một số điều kiện, không thay thế việc xem các trường hợp sai.
- Với các phần có hướng dẫn, tự viết trước khi nhờ AI.
  Các bài kiểm tra trên lớp (🚫 đóng) đánh giá các kỹ năng này.
- Bài tự làm ✅ mở cho phép dùng AI, kèm khai báo và cách kiểm chứng.
- Khi cần hỗ trợ, trao đổi với giảng viên thực hành về yêu cầu hoặc trường hợp chưa hiểu.

## Mục tiêu

1. Sắp xếp một đoạn báo cáo theo thứ tự kết luận, bằng chứng, giới hạn, cách làm.
2. Phân biệt lời diễn đạt đúng mức và quá mức với số liệu thật: thước đo, độ lớn của con số, cỡ mẫu.
3. Tách một con số gộp theo nhóm bằng `groupby` và `crosstab` để thấy cơ cấu nhóm làm đổi con số đó.
4. Thẩm định một kết luận bằng bốn bước: tính lại số liệu, kiểm tra phương pháp, kiểm tra diễn giải,
   rồi kết luận giữ, sửa hoặc chưa đủ căn cứ.

## Chuẩn bị

Dữ liệu thật là Santiago, mốc chụp **29/06/2026**, ba file:

- `visualisations/listings.csv`: mỗi hàng là một chỗ ở; `price` là số, đơn vị **CLP/đêm**; `room_type` là loại phòng;
  `minimum_nights` là số đêm thuê tối thiểu; `calculated_host_listings_count` là số chỗ ở của cùng chủ nhà
  trong mốc chụp; `number_of_reviews_ltm` là số đánh giá trong 12 tháng trước mốc chụp.
- `visualisations/reviews.csv`: mỗi hàng là một đánh giá, gồm `listing_id` (mã chỗ ở) và `date`.
- `data/listings.csv.gz`: bản đầy đủ 90 cột. Ô chuẩn bị chỉ đọc hai cột, `id` và `review_scores_rating`
  (điểm đánh giá của chỗ ở, thang 5 điểm; trống với chỗ ở chưa có đánh giá).

Ô chuẩn bị cắt `rv` theo mốc chụp và tạo cột `kieu_host` trong `df`: "chuyên nghiệp" nếu chủ nhà có từ 5 chỗ ở
trong mốc chụp, như Lab 13, ngược lại là "cá nhân". Hàm `bo_dau` chuẩn hoá câu trả lời dạng chữ (bỏ dấu, chữ thường),
nên viết "vừa" hay "vua" đều được; hàm `ma` cho phép ô kiểm tra so sánh câu trả lời mà notebook không chứa sẵn đáp án.

Dùng file cục bộ nếu có; nếu chưa có thì đọc đúng mốc chụp từ nguồn.
Chạy hai ô dưới trước. Nếu phiên Colab đã nạp pandas phiên bản khác, chọn
**Restart session and run all** sau khi cài.

In [ ]:
%pip -q install pandas==3.0.3

In [ ]:
import hashlib
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

GOC = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/"
tep_ls = Path("data/listings_santiago_2026-06-29.csv")
tep_rv = Path("data/reviews_santiago_2026-06-29.csv")
tep_full = Path("data/listings_santiago_full_2026-06-29.csv.gz")
df = pd.read_csv(tep_ls if tep_ls.exists() else GOC + "visualisations/listings.csv")
rv = pd.read_csv(tep_rv if tep_rv.exists() else GOC + "visualisations/reviews.csv",
                 parse_dates=["date"])
rv = rv[rv["date"] <= "2026-06-29"]                  # cắt theo mốc chụp (Bài 8)
full = pd.read_csv(tep_full if tep_full.exists() else GOC + "data/listings.csv.gz",
                   usecols=["id", "review_scores_rating"])   # chỉ đọc hai cột cần dùng

df["kieu_host"] = (df["calculated_host_listings_count"] >= 5).map(
    {True: "chuyên nghiệp", False: "cá nhân"})


def bo_dau(chu):
    """Chuẩn hoá câu trả lời dạng chữ: bỏ dấu, chữ thường, khoảng trắng thành "_"."""
    chu = unicodedata.normalize("NFD", str(chu).replace("đ", "d").replace("Đ", "D"))
    chu = "".join(c for c in chu if unicodedata.category(c) != "Mn")
    return "_".join(chu.lower().split())


def ma(chu):
    """Mã băm ngắn của câu trả lời đã chuẩn hoá, dùng trong các ô kiểm tra."""
    return hashlib.sha256(bo_dau(chu).encode()).hexdigest()[:10]


print("pandas:", pd.__version__)
print("Số chỗ ở:", len(df), "| Số đánh giá:", len(rv), "| Bản đầy đủ:", len(full), "hàng")
print(df["kieu_host"].value_counts())

## Phần 1 · Đặt kết luận lên trước

Bốn câu A–D dưới đây thuộc một đoạn báo cáo về giá ở Santiago, viết cho chủ nhà mới, nhưng đang bị xáo trộn.
Các con số trong câu tính từ dữ liệu của lab này. Sắp xếp lại theo thứ tự
**kết luận → bằng chứng → giới hạn → cách làm**:

- Kết luận đứng đầu, vì người đọc cần câu trả lời trước (slide «Đặt kết luận lên trước» ở Bài 14);
  câu này chính là ý chính của đoạn (slide «Nêu ý chính trong một câu»).
- Giới hạn đi ngay sau bằng chứng, trong cùng đoạn với kết luận: người đọc biết con số đúng trong phạm vi nào
  ngay khi đọc nó, kể cả khi chỉ đọc đoạn đầu (slide «Nêu giới hạn cạnh kết luận»).
  Dồn giới hạn xuống cuối báo cáo thì người đọc dễ bỏ qua.
- Cách làm đứng cuối hoặc ở phụ lục: chỉ người cần kiểm tra lại mới cần đọc.

In [ ]:
CAU = {
    "A": "Bỏ 177 giá cao hơn phân vị 99 (686.144 CLP/đêm) thì giá trung bình giảm còn 82.298 CLP/đêm, "
         "còn giá trung vị chỉ đổi từ 59.000 xuống 58.400 CLP/đêm.",
    "B": "Nhóm đọc listings.csv của mốc chụp 29/06/2026, bỏ các giá trống bằng dropna, "
         "rồi tính mean, median và quantile(0.99) trên cột price.",
    "C": "Giá thuê phổ biến ở Santiago (giá trung vị 59.000 CLP/đêm) chỉ bằng một nửa "
         "giá trung bình 118.200 CLP/đêm.",
    "D": "Các con số chỉ tính trên 17.688 chỗ ở có giá, chưa gồm 846 chỗ ở thiếu giá; nhóm giá rất cao "
         "có giá bất thường chưa được xác minh, như mức 97.000.045 CLP/đêm.",
}
for k, v in CAU.items():
    print(f"{k}. {v}\n")

In [ ]:
# TODO: thứ tự bốn chữ cái, dạng list (vd ["A", "B", "C", "D"]) hoặc chuỗi (vd "ABCD").
thu_tu = None

In [ ]:
assert thu_tu is not None, "Điền thu_tu: thứ tự bốn chữ cái A–D."
assert isinstance(thu_tu, (list, tuple, str)), (
    'thu_tu là một list (vd ["A", "B", "C", "D"]) hoặc một chuỗi (vd "ABCD").')
if isinstance(thu_tu, str):
    tt = [c for c in thu_tu.upper() if c.isalpha()]
else:
    tt = [str(c).strip().upper() for c in thu_tu]
assert sorted(tt) == ["A", "B", "C", "D"], "Dùng đủ bốn chữ A, B, C, D, mỗi chữ đúng một lần."
KHOA_1 = ['e16b12e848', 'e9c3b03775', '62399f8e52', 'c4d197346c']
dung = [ma(f"p1-{i}-{c}") == KHOA_1[i] for i, c in enumerate(tt)]
assert dung[0], f"Câu {tt[0]} chưa phải kết luận. Câu đầu tiên nêu phát hiện chính, kèm con số."
assert dung[3], f"Câu {tt[3]} chưa phải cách làm. Câu cuối kể các bước đã làm: đọc file nào, dùng hàm nào."
assert dung[1] and dung[2], "Câu thứ hai là bằng chứng cho kết luận; câu thứ ba nêu giới hạn của các con số."
print("Đúng thứ tự:", " → ".join(tt))

**Nhận xét của bạn:** Câu giới hạn nêu những giới hạn nào của các con số? Nếu chuyển câu đó xuống cuối báo cáo,
người đọc chỉ đọc đoạn đầu có thể hiểu sai điều gì về mức 118.200 CLP/đêm? …

## Phần 2 · Diễn đạt đúng mức

Mỗi cặp dưới đây gồm số liệu và một lời diễn đạt viết từ số liệu đó. Gán `"vua"` nếu lời diễn đạt đúng mức
với số liệu, `"qua"` nếu quá mức (slide «Diễn đạt đúng mức» ở Bài 14). Với mỗi cặp, xem ba điều:

- lời diễn đạt có nói về đúng đại lượng mà số liệu đo không;
- từ ngữ có hợp với độ lớn của con số không;
- cỡ mẫu có đủ cho điều lời diễn đạt khẳng định không.

Ô đầu in sáu cặp; ô thứ hai tính lại các số liệu từ dữ liệu.

In [ ]:
CAP = {
    1: ("Số đánh giá tháng 5/2026 là 22.296, tháng 5/2025 là 14.509 (+53,7%).",
        "Số đánh giá tháng 5/2026 tăng khoảng 54% so với tháng 5/2025."),
    2: ("Số đánh giá tháng 5/2026 là 22.296, tháng 5/2025 là 14.509 (+53,7%).",
        "Nhu cầu thuê chỗ ở Airbnb ở Santiago tăng 54% trong một năm."),
    3: ("Số đánh giá tháng 5/2026 là 22.296, tháng 4/2026 là 21.656 (+3,0%).",
        "Số đánh giá bứt phá trong tháng 5/2026."),
    4: ("Khách sạn: 42 chỗ ở của chủ nhà chuyên nghiệp, giá trung vị 116.213 CLP/đêm; "
        "14 chỗ ở của chủ nhà cá nhân, giá trung vị 123.578,5 CLP/đêm.",
        "Chủ nhà cá nhân định giá khách sạn cao hơn hẳn chủ nhà chuyên nghiệp."),
    5: ("Giá trung bình 118.200 CLP/đêm, giá trung vị 59.000 CLP/đêm (17.688 chỗ ở có giá).",
        "Giá trung bình gấp khoảng hai lần giá trung vị: phân phối giá lệch phải, "
        "một số giá rất cao kéo mức trung bình lên."),
    6: ("1 trong 18.534 chỗ ở yêu cầu thuê tối thiểu 730 đêm.",
        "Có một chỗ ở với số đêm tối thiểu bất thường (730 đêm), cần gắn cờ để kiểm tra."),
}
for k, (so_lieu, loi) in CAP.items():
    print(f"{k}. Số liệu: {so_lieu}\n   Lời diễn đạt: {loi}\n")

In [ ]:
thang = rv.set_index("date").resample("ME").size()
for t in ["2025-05", "2026-04", "2026-05"]:
    print(f"số đánh giá {t}: {thang.loc[t].iloc[0]}")
gia = df["price"].dropna()
print(f"{len(gia)} chỗ ở có giá: trung bình {gia.mean():.0f}, trung vị {gia.median():.0f} CLP/đêm")
print("số chỗ ở có minimum_nights == 730:", (df["minimum_nights"] == 730).sum())
print(df[df["room_type"] == "Hotel room"].groupby("kieu_host")["price"].agg(["median", "size"]))

In [ ]:
# TODO: gán "vua" hoặc "qua" cho từng cặp: nhan = {1: "...", 2: "...", ..., 6: "..."}.
nhan = None

In [ ]:
assert nhan is not None, 'Điền nhan: mỗi cặp một nhãn "vua" hoặc "qua".'
assert isinstance(nhan, dict), 'nhan là một dict, vd {1: "vua", 2: "qua", ...}.'
assert all(str(k).strip().isdigit() for k in nhan), "Khoá của nhan là số thứ tự của cặp, từ 1 đến 6."
nhan_cd = {int(k): bo_dau(v) for k, v in nhan.items()}
assert set(nhan_cd) == set(CAP), "Gán nhãn cho đủ sáu cặp, khoá là số thứ tự từ 1 đến 6."
assert set(nhan_cd.values()) <= {"vua", "qua"}, 'Mỗi nhãn là "vua" (đúng mức) hoặc "qua" (quá mức).'
KHOA_2 = {1: '91d848bfa5', 2: '1ea284af1c', 3: 'ea8a9a6b28', 4: 'f0291eba56', 5: '7cbd8c5aae', 6: '2472e4465b'}
GOI_Y = {
    1: "Cặp 1: so sánh đại lượng, kỳ so sánh và độ lớn trong lời diễn đạt với số liệu.",
    2: "Cặp 2: số liệu đếm đánh giá; lời diễn đạt nói về đại lượng nào? "
       "Xem slide «Nêu giới hạn cạnh kết luận».",
    3: "Cặp 3: mức tăng 3,0% so với tháng trước lớn hay nhỏ so với từ ngữ được dùng?",
    4: "Cặp 4: xem cỡ mẫu của từng nhóm trước khi đánh giá lời diễn đạt.",
    5: "Cặp 5: lời diễn đạt có nói nhiều hơn điều hai con số cho thấy không?",
    6: "Cặp 6: lời diễn đạt có nói nhiều hơn điều số liệu cho thấy không?",
}
sai = [k for k in sorted(CAP) if ma(f"p2-{k}-{nhan_cd[k]}") != KHOA_2[k]]
assert not sai, "Chưa đúng ở " + str(len(sai)) + " cặp:\n" + "\n".join(GOI_Y[k] for k in sai)
print("Đúng cả sáu cặp.")

**Nhận xét của bạn:** Với mỗi cặp bạn gán "qua", lời diễn đạt sai ở đâu: đại lượng, độ lớn hay cỡ mẫu?
Viết lại lời diễn đạt của cặp 2 cho đúng mức, kèm một giới hạn của số đánh giá. …

## Phần 3 · Cơ cấu nhóm làm thay đổi con số gộp

Câu hỏi: chỗ ở của chủ nhà chuyên nghiệp có nhận nhiều đánh giá hơn chỗ ở của chủ nhà cá nhân không?
Thước đo là `number_of_reviews_ltm`, số đánh giá trong 12 tháng trước mốc chụp.
Ở slide «Cơ cấu nhóm làm thay đổi con số gộp» của Bài 14, chỗ ở có chữ "metro" trong tên tập trung ở các quận rẻ,
nên giá gộp của nhóm này thấp hơn, dù so sánh trong cùng quận thì hai nhóm gần như bằng nhau.
Phần này làm tương tự cho hai nhóm chủ nhà, tách theo loại phòng.

**a)** Tạo `gop`: trung bình `number_of_reviews_ltm` theo `kieu_host`, tính trên mọi chỗ ở của `df`
(một Series, chỉ mục là hai nhóm chủ nhà).

In [ ]:
# TODO: tạo gop.
gop = None

In [ ]:
assert gop is not None, "Tạo gop."
assert isinstance(gop, pd.Series), (
    'gop là Series: groupby("kieu_host"), chọn cột ["number_of_reviews_ltm"] rồi mới tính mean().')
assert set(gop.index) == {"chuyên nghiệp", "cá nhân"}, (
    "Nhóm theo cột kieu_host: chỉ mục của gop là hai nhóm chủ nhà.")
assert gop.max() < 30, (
    "Dùng number_of_reviews_ltm (12 tháng trước mốc chụp), không dùng number_of_reviews (tính từ trước tới nay).")
assert not (gop["chuyên nghiệp"] == 7 and gop["cá nhân"] == 4), "Tính trung bình (mean), không phải trung vị."
assert abs(gop["cá nhân"] - 13.13) > 0.01, "Tính trên mọi chỗ ở của df (18.534), kể cả chỗ ở thiếu giá."
assert abs(gop["chuyên nghiệp"] - 15.51) < 0.06 and abs(gop["cá nhân"] - 12.65) < 0.06, (
    "gop là trung bình number_of_reviews_ltm của mọi chỗ ở trong df, theo kieu_host.")
print(f"Số đánh giá trung bình trong 12 tháng: chuyên nghiệp {gop['chuyên nghiệp']:.1f}, "
      f"cá nhân {gop['cá nhân']:.1f}")

**b)** Tạo `co_cau`: tỷ trọng từng loại phòng trong mỗi nhóm chủ nhà, **dạng tỷ lệ từ 0 đến 1**.
Mỗi hàng là một nhóm chủ nhà, mỗi cột là một loại phòng, mỗi hàng cộng lại bằng 1
(`pd.crosstab(..., normalize="index")`, Bài 10).

In [ ]:
# TODO: tạo co_cau.
co_cau = None

In [ ]:
assert co_cau is not None, "Tạo co_cau."
assert isinstance(co_cau, pd.DataFrame), "co_cau là DataFrame: dùng pd.crosstab(...)."
assert "chuyên nghiệp" in co_cau.index, (
    "Mỗi hàng là một nhóm chủ nhà: đặt df[\"kieu_host\"] làm đối số thứ nhất của crosstab.")
assert "Entire home/apt" in co_cau.columns, "Mỗi cột là một loại phòng (room_type)."
assert co_cau.to_numpy().max() <= 1, 'co_cau là tỷ lệ từ 0 đến 1: thêm normalize="index", không nhân 100.'
assert np.allclose(co_cau.sum(axis=1), 1), (
    'Mỗi hàng cộng lại bằng 1: dùng normalize="index" (tỷ trọng trong từng nhóm chủ nhà).')
assert abs(co_cau.loc["chuyên nghiệp", "Entire home/apt"] - 0.8759) < 0.001, (
    "Tính trên mọi chỗ ở của df (18.534), kể cả chỗ ở thiếu giá.")
print(co_cau[["Entire home/apt", "Private room"]].round(3))

**c)** Tạo `tach`: trung bình `number_of_reviews_ltm` theo cả loại phòng và nhóm chủ nhà.
Mỗi hàng là một loại phòng, mỗi cột là một nhóm chủ nhà (`groupby` theo hai cột rồi `unstack()`, Bài 5).

In [ ]:
# TODO: tạo tach.
tach = None

In [ ]:
assert tach is not None, "Tạo tach."
assert isinstance(tach, pd.DataFrame), (
    "tach là DataFrame: groupby theo hai cột, chọn cột number_of_reviews_ltm, tính mean() rồi unstack().")
assert "Entire home/apt" in tach.index and "chuyên nghiệp" in tach.columns, (
    "Mỗi hàng là một loại phòng, mỗi cột là một nhóm chủ nhà: groupby([\"room_type\", \"kieu_host\"]) rồi unstack().")
assert len(tach) == 4, "tach có bốn hàng, mỗi loại phòng một hàng."
assert tach.to_numpy().max() < 30, (
    "Dùng number_of_reviews_ltm (12 tháng trước mốc chụp), không dùng number_of_reviews.")
chinh = tach.loc[["Entire home/apt", "Private room"], ["chuyên nghiệp", "cá nhân"]]
assert np.allclose(chinh.to_numpy(), [[16.870, 14.576], [6.093, 6.118]], atol=0.006), (
    "tach là trung bình (mean) number_of_reviews_ltm của mọi chỗ ở trong df, theo room_type và kieu_host.")
print(chinh.round(2))
print("\nSố chỗ ở theo loại phòng và nhóm chủ nhà:")
print(df.groupby(["room_type", "kieu_host"]).size().unstack())

Ở cả hai nhóm, nguyên căn nhận nhiều đánh giá hơn phòng riêng (khoảng 15–17 so với 6 đánh giá trong 12 tháng).
Khách sạn và phòng chung có ít chỗ ở (42 và 14; 33 và 28), chưa đủ để so sánh hai nhóm.

**Nhận xét của bạn:** Chênh lệch gộp giữa hai nhóm là khoảng 2,9 đánh giá. Khi tách theo loại phòng,
chênh lệch còn lại bao nhiêu ở nguyên căn và ở phòng riêng? Dùng `co_cau` để giải thích vì sao con số gộp
lớn hơn chênh lệch trong phòng riêng. Viết hai câu cho báo cáo: một câu kết luận đã tách theo loại phòng,
một câu giới hạn. …

## Phần 4 · Thẩm định một kết luận

Câu dưới là **câu soạn minh hoạ** theo kiểu một báo cáo do AI viết, không phải output thật của mô hình.
Thẩm định câu này bằng bốn bước ở slide «Bốn bước thẩm định» của Bài 14:
tính lại số liệu → kiểm tra phương pháp → kiểm tra diễn giải → kết luận giữ, sửa hoặc chưa đủ căn cứ.

In [ ]:
CAU_AI = ("Chủ nhà chuyên nghiệp được khách yêu thích hơn: họ nhận nhiều đánh giá hơn "
          "(15,5 so với 12,7 đánh giá trong 12 tháng).")
print(CAU_AI)

**Bước 1 · Tính lại số liệu.** Hai con số trong câu là `gop` ở Phần 3a. Ô dưới in lại, làm tròn một chữ số thập phân.

In [ ]:
assert gop is not None, "Làm Phần 3a trước: tạo gop."
print(gop.round(1))

**Bước 2 · Kiểm tra phương pháp.** Thước đo là số đánh giá trong 12 tháng, gộp mọi loại phòng.
Phần 3 cho biết hai nhóm có cơ cấu loại phòng khác nhau (`co_cau`) và chênh lệch còn lại trong từng loại phòng
(`tach`), kèm cỡ mẫu của từng ô. Ghi kết quả của bước này vào ô nhận xét cuối phần.

**Bước 3 · Kiểm tra diễn giải.** "Được khách yêu thích" nói về việc khách đánh giá chất lượng chỗ ở cao hay thấp,
không phải số lượng đánh giá. Trong dữ liệu, cột gần nhất với ý này là `review_scores_rating` của bản đầy đủ (`full`).

**a)** Tạo `ghep`: ghép cột `review_scores_rating` của `full` vào `df` theo `id` (`merge`, Bài 5),
giữ đủ 18.534 chỗ ở của `df`. Tạo `diem` từ `ghep`: mỗi nhóm chủ nhà một hàng, hai cột `mean` (điểm trung bình)
và `count` (số chỗ ở có điểm), bằng `groupby("kieu_host")[...].agg(["mean", "count"])`.

In [ ]:
# TODO: tạo ghep và diem.
ghep = None
diem = None

In [ ]:
assert ghep is not None and diem is not None, "Tạo ghep và diem."
assert isinstance(ghep, pd.DataFrame), "ghep là DataFrame: df.merge(full, ...)."
assert {"kieu_host", "review_scores_rating"} <= set(ghep.columns), (
    "ghep cần cả cột kieu_host (của df) và review_scores_rating (của full): ghép df.merge(full, on=\"id\").")
assert len(ghep) == len(df), "ghep giữ đủ 18.534 chỗ ở của df, mỗi chỗ ở một hàng: ghép theo id (on=\"id\")."
assert isinstance(diem, pd.DataFrame), 'diem là DataFrame: groupby("kieu_host")["review_scores_rating"].agg(["mean", "count"]).'
assert {"mean", "count"} <= set(diem.columns), "diem cần hai cột mean và count."
assert set(diem.index) == {"chuyên nghiệp", "cá nhân"}, "Mỗi hàng của diem là một nhóm chủ nhà (kieu_host)."
assert diem["count"].sum() != len(df), (
    "count là số chỗ ở có điểm; chỗ ở chưa có đánh giá thì điểm trống và không được đếm. "
    "Dùng cột review_scores_rating, không đếm bằng size.")
assert diem.loc["chuyên nghiệp", "count"] == 5644 and diem.loc["cá nhân", "count"] == 9599, (
    "count đếm số chỗ ở có review_scores_rating trong từng nhóm chủ nhà.")
assert abs(diem.loc["chuyên nghiệp", "mean"] - 4.731) < 0.005 and abs(diem.loc["cá nhân", "mean"] - 4.817) < 0.005, (
    "mean là điểm trung bình review_scores_rating của từng nhóm chủ nhà.")
print(diem.round(2))
print("\nĐiểm trung bình theo loại phòng:")
print(ghep.groupby(["room_type", "kieu_host"])["review_scores_rating"].mean()
      .unstack().loc[["Entire home/apt", "Private room"]].round(2))

Điểm đánh giá dồn về gần 5: trung vị của mọi chỗ ở có điểm là 4,88. Trên thang đó, điểm trung bình của nhóm
chuyên nghiệp thấp hơn nhóm cá nhân, và thấp hơn ở cả nguyên căn lẫn phòng riêng.

**Bước 4 · Kết luận.** Chọn `ket_luan` là một trong ba giá trị `"giu"`, `"sua"`, `"chua_du_can_cu"`
(viết có dấu cũng được), theo cách xếp loại ở slide «Bảng kết quả thẩm định» của Bài 14.
Viết vào `cau_moi` câu bạn sẽ đưa vào báo cáo thay cho `CAU_AI`: giữ những con số đứng vững,
bỏ phần vượt quá dữ liệu.

In [ ]:
# TODO: chọn ket_luan và viết cau_moi.
ket_luan = None
cau_moi = None

In [ ]:
assert ket_luan is not None and cau_moi is not None, "Điền ket_luan và cau_moi."
assert isinstance(ket_luan, str), 'ket_luan là một chuỗi: "giu", "sua" hoặc "chua_du_can_cu".'
assert bo_dau(ket_luan) in {"giu", "sua", "chua_du_can_cu"}, (
    'ket_luan là một trong ba giá trị "giu", "sua", "chua_du_can_cu".')
GOI_Y_4 = {
    '2642d5234d': "Giữ nguyên văn nghĩa là số liệu, phương pháp và lời diễn giải đều đứng vững. "
                  "Đối chiếu lời \"được khách yêu thích hơn\" với kết quả Bước 3.",
    '1f2d228c69': "Chưa đủ căn cứ dùng khi chính phép so sánh không hợp lệ, như ý 3 ở Bài 14 (so sánh tháng chưa trọn vẹn "
                  "với tháng trọn vẹn). Ở đây hai con số tính trên cùng mốc chụp, cùng 12 tháng; xem lại lỗi nằm ở bước nào.",
}
assert ma(f"p4-{bo_dau(ket_luan)}") == '839a5183c7', GOI_Y_4.get(ma(f"p4-{bo_dau(ket_luan)}"), "")
assert isinstance(cau_moi, str) and len(cau_moi.strip()) >= 40, "cau_moi là một câu đầy đủ (chuỗi, ít nhất 40 ký tự)."
assert cau_moi.strip() != CAU_AI, "Viết lại câu, không chép nguyên văn CAU_AI."
assert any(c.isdigit() for c in cau_moi), "Câu mới vẫn nêu con số, kèm phép so sánh."
print("Kết luận thẩm định:", ket_luan)
print("Câu mới:", cau_moi.strip())

**Nhận xét của bạn:** Ghi kết quả từng bước: (1) hai con số có đúng không; (2) sau khi tách theo loại phòng,
chênh lệch còn ở đâu, và cỡ mẫu nào quá nhỏ; (3) lời "được khách yêu thích" khớp với thước đo nào,
và điểm đánh giá cho thấy điều gì; (4) vì sao bạn chọn kết luận đó. Câu mới của bạn nêu giới hạn nào? …

## Phần 5 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

### Tự làm 1 · Thẩm định một đoạn trong báo cáo của nhóm

Chọn một đoạn có ít nhất ba ý trong bản nháp báo cáo bài tập lớn của nhóm, hoặc nhờ một công cụ AI viết
một đoạn từ bảng chỉ số của nhóm, như yêu cầu gửi cho Gemini ở Bài 14. Thẩm định từng ý bằng bốn bước:
tính lại ít nhất một con số bằng code trong ô dưới; kiểm tra phương pháp (thước đo, mốc so sánh, tháng trọn vẹn, cỡ mẫu);
kiểm tra diễn giải. Ghi kết luận giữ, sửa hoặc chưa đủ căn cứ cho từng ý, kèm lý do, theo mẫu
slide «Bảng kết quả thẩm định».

In [ ]:
# Viết bài tự làm 1 ở đây

**Nhận xét của bạn:** …

### Tự làm 2 · Viết lại đoạn mở đầu báo cáo của nhóm

Viết lại đoạn mở đầu báo cáo của nhóm: câu đầu nêu ý chính (slide «Nêu ý chính trong một câu»),
kết luận đặt trước bằng chứng (slide «Đặt kết luận lên trước»), con số đặt trong phép so sánh,
giới hạn đi ngay cạnh kết luận. Ghi rõ người đọc (bên đặt hàng nhóm đã chọn) và mốc chụp.
Đặt đoạn cũ cạnh đoạn mới, rồi ghi ba thay đổi chính.

**Đoạn cũ:** …

**Đoạn mới:** …

**Ba thay đổi chính:** …

**Khai báo AI (nếu dùng):** công cụ, yêu cầu chính, một kết quả bạn đã tự kiểm tra. …

## Hỗ trợ bài tập lớn (làm theo nhóm)

Trọng tâm tuần này: **rà soát lần cuối trước khi nộp bản cuối bằng `git tag final`,
hạn 23:59 Chủ nhật 22/11/2026 (tuần 11).** Danh sách đầy đủ ở đề bài tập lớn.

1. ☐ Bài kiểm tra "máy sạch": một thành viên `git clone` repo vào thư mục mới, dựng môi trường theo README
   (Python theo `.python-version`, cài `requirements.txt`), chạy quy trình bằng một lệnh,
   rồi đối chiếu số liệu và hình tạo lại với báo cáo.
2. ☐ Chạy thử quy trình trên một mốc chụp nhóm chưa dùng, chỉ sửa tệp cấu hình trong `configs/`.
3. ☐ `requirements.txt` ghim phiên bản bằng `==`; `.python-version` ghi một dòng phiên bản Python đã chạy thử.
4. ☐ Khoá API đọc từ biến môi trường và không có trong repo, kể cả lịch sử commit:
   `git log -p --all | grep "AIza"` không ra kết quả (thay `AIza` bằng vài ký tự đầu của khoá nhóm dùng);
   `.env` nằm trong `.gitignore`, `.env.example` chỉ có tên biến.
5. ☐ README ghi cách cài, lệnh chạy, tên biến môi trường chứa khoá API và tệp kết quả để đối chiếu.
6. ☐ `AI_USAGE.md` đủ bốn mục; mục "AI sai ở đâu" có ví dụ thật và cách nhóm phát hiện.
7. ☐ `data/samples/` có ít nhất 100 nhãn tay; `figures/` sinh tự động; `reports/` có báo cáo PDF và `qa_report.csv`.
8. ☐ Báo cáo: đoạn mở đầu nêu ý chính, kết luận đặt trước; mỗi kết luận có giới hạn đi kèm; tiêu đề hình nêu kết luận;
   ghi nguồn và ngày của từng mốc chụp. Mọi thành viên truy lại được từng con số trong báo cáo về code và dữ liệu.
9. ☐ Gắn tag `final` và đẩy lên (`git push origin final`); ghi mã commit vào báo cáo.

Nhóm xong sớm: thẩm định chéo hai kết luận trong báo cáo của nhóm bên cạnh bằng bốn bước;
ghi kết luận giữ, sửa hoặc chưa đủ căn cứ cho từng kết luận, kèm lý do, rồi gửi lại cho nhóm đó.

## Tóm tắt bài lab

- Một đoạn báo cáo đi theo thứ tự kết luận → bằng chứng → giới hạn → cách làm; giới hạn đứng cạnh kết luận,
  cách làm để cuối hoặc ở phụ lục.
- Lời diễn đạt đúng mức nói về đúng đại lượng được đo (số đánh giá, không phải nhu cầu thuê),
  dùng từ hợp với độ lớn của con số và với cỡ mẫu.
- Con số gộp phụ thuộc cơ cấu nhóm: tách theo loại phòng trước khi so sánh hai nhóm chủ nhà.
- Tính lại số liệu chỉ là bước đầu của thẩm định: số đúng mà lời diễn giải vẫn có thể sai,
  và cần kiểm tra bằng một thước đo khớp với lời diễn giải.

Chạy **Restart session and run all**, rồi kiểm tra: mọi ô code đều có kết quả; các ô
nhận xét và khai báo AI (nếu dùng) đã điền đủ.

**Đọc thêm:** [Wilke, Fundamentals of Data Visualization, chương 29](https://clauswilke.com/dataviz/telling-a-story.html),
[Calling Bullshit](https://callingbullshit.org/syllabus.html) (Bergstrom và West), tuần 5 (Statistical traps and trickery)
và tuần 12 (Refuting bullshit).

**Bài sau:** vấn đáp bài tập lớn (Bài 15).